# 📦 Deployment Pipelines Notebook

This notebook deploys artifacts in a selected deployment pipeline from Source → Target using Microsoft Fabric APIs.  

---

## 📈 Steps
1️⃣ Set Parameters  
2️⃣ Get the cutoff date for deployment  
3️⃣ Construct the API call with selected parameters  
4️⃣ Make the API call  

---

## 🎯 Notes
- This notebook uses Semantic Link Labs wrapper functions. It is recommended to use a pre-configured environment with the libraries installed. Otherwise, semantic-link-labs needs to be pip installed 
- In this current version, the item_types parameter is mandatory. A sample of acceptable common item types are as follows:
    - Lakehouse, Warehouse, SQLDatabase, SemanticModel, Report, SQLEndpoint, DataPipeline, Notebook  
    - The full list of acceptable items types can be found here: https://learn.microsoft.com/en-us/rest/api/fabric/admin/items/list-items?tabs=HTTP#itemtype 

## 🔧 Install Dependencies

In [ ]:
# Not needed with pre-configured environment
# %pip install semantic-link-labs

In [ ]:
import sempy_labs as labs
import sempy_labs.admin as admin
import requests
import pandas as pd
import time

## 🔧 Parameters

In [ ]:
# 🔧 Parameters
p_modified_after = '2025-07-08'                             # Will deploy artifacts modified after this date. If left blank will default to max date in the target stage
p_deployment_pipeline_name = "Data Engineering Pipeline"    # Name of the deployment pipeline
p_source_stage_name = "Dev"                                 # Name of the stage in the deployment pipeline to deploy from
p_target_stage_name = "Test"                                # Name of the stage in the deployment pipeline to deploy to
p_workspace_name = "Data Engineering [Dev]"                 # Source workspace, used to retrieve item types to deploy
p_item_types = ["SQLDatabase","SQLEndpoint"]      # Items types to deploy

## 🕒 Get deployment cutoff date

In [ ]:
# 🕒 Determine cutoff_date
if p_modified_after.strip():
    cutoff_date = pd.to_datetime(p_modified_after)
else:
    # Get deployment history for the target stage
    stage_items_df = labs.list_deployment_pipeline_stage_items(
        p_deployment_pipeline_name, p_target_stage_name
    )
    # Convert to datetime
    stage_items_df["Last Deployment Time"] = pd.to_datetime(
        stage_items_df["Last Deployment Time"], format='mixed', errors='coerce'
    )
    
    # Use the most recent deployment time, or fallback to a very early date
    if not stage_items_df["Last Deployment Time"].isna().all():
        cutoff_date = stage_items_df["Last Deployment Time"].max()
    else:
        cutoff_date = pd.Timestamp("2025-01-01")

print(f"Using cutoff date: {cutoff_date}")

## 🧰 Build Items Payload for API Call

In [ ]:
# 📋 Retrieve pipeline stages
stages_df = labs.list_deployment_pipeline_stages(p_deployment_pipeline_name)

# 🔍 Resolve stage IDs
def get_stage_id(stage_name):
    match = stages_df.loc[stages_df["Deployment Pipeline Stage Name"] == stage_name, "Deployment Pipeline Stage Id"]
    if match.empty:
        raise ValueError(f"Stage '{stage_name}' not found.")
    return match.values[0]

source_stage_id = get_stage_id(p_source_stage_name)
target_stage_id = get_stage_id(p_target_stage_name)

# 🧭 Resolve workspace ID from name
workspace_id = stages_df.loc[stages_df["Workspace Name"] == p_workspace_name, "Workspace Id"].values[0]

# 📦 List items in workspace and filter by type
workspace_items_df = admin.list_items(workspace=workspace_id)

# 🕒 Convert to datetime
workspace_items_df["Last Updated Date"] = pd.to_datetime(workspace_items_df["Last Updated Date"])

# 🔍 Filter by item type and last updated date
filtered_items = workspace_items_df[
    (workspace_items_df["Type"].isin(p_item_types)) &
    (workspace_items_df["Last Updated Date"] > cutoff_date)
]

# 🧾 Build items array
items_payload = [
    {"sourceItemId": row["Item Id"], "itemType": row["Type"]}
    for _, row in filtered_items.iterrows()
]

print(f"Source stage id: {source_stage_id}")
print(f"Target stage id: {target_stage_id}")
display(items_payload)

### 📌 REST Setup

In [ ]:
# Get Access Token and Setup Session
api_access_token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
if not api_access_token:
    raise Exception("Failed to get access token")

def create_fabric_session(fabric_token):
    fabric_headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {fabric_token}'
    }
    session = requests.Session()
    session.headers.update(fabric_headers)
    return session

session = create_fabric_session(api_access_token)

## 🚀 Make the API call

In [ ]:
# 🧰 Final payload
payload = {
    "sourceStageId": source_stage_id,
    "targetStageId": target_stage_id,
    "items": items_payload,
    "note": "Deploying selected items by type"
}

# 🌐 API endpoint
deployment_pipelines = labs.list_deployment_pipelines() 
pipeline_id = deployment_pipelines.loc[
    deployment_pipelines["Deployment Pipeline Name"] == p_deployment_pipeline_name,
    "Deployment Pipeline Id"
].values[0]

url = f"https://api.fabric.microsoft.com/v1/deploymentPipelines/{pipeline_id}/deploy"

# 🚀 Make the API call
response = session.post(url, json=payload)

# ✅ Extract operationId from response headers
operation_id = response.headers.get("x-ms-operation-id")

if operation_id:
    print(f"✅ Deployment initiated. Operation ID: {operation_id}")

    # 📡 Construct status URL
    status_url = f"https://api.fabric.microsoft.com/v1/deploymentPipelines/{pipeline_id}/operations/{operation_id}"

    # ⏳ Poll for status
    retry_after = int(response.headers.get("Retry-After", 10))
    status_data = None
    status = None

    for attempt in range(15):
        status_response = session.get(status_url)
        if status_response.status_code == 200:
            status_data = status_response.json()
            status = status_data.get("status")
            print(f"🔄 Attempt {attempt+1}: Status = {status}")
            if status in ["Succeeded", "Failed"]:
                break
        elif status_response.status_code == 404:
            print(f"🔄 Attempt {attempt+1}: Status endpoint not ready yet. Retrying in {retry_after} seconds.")
            time.sleep(retry_after)
            continue
        else:
            print(f"⚠️ Unexpected status code: {status_response.status_code}")
            break

    # 📋 If failed, show detailed info
    if status == "Failed" and status_data:
        print("❌ Deployment failed. Details:")
        print(status_data)
    elif status == "Succeeded":
        print("✅ Deployment completed successfully.")
    elif status is None:
        print("⚠️ Status could not be retrieved after multiple attempts.")
else:
    print("⚠️ Deployment initiated, but operationId not found in headers.")